# Day 55 Project — Solution: Auth AI API

Full auth flow: register → login → JWT → protected routes.

**Deliverable:** `auth_api.py` — run with `uvicorn auth_api:app --reload`.

In [ ]:
# ── provided source string ──────────────────────────────────────────────────
_AUTH_API_SRC = '"""auth_api.py — Day 055 project: FastAPI with password auth + JWT.\n\nRun:  uvicorn auth_api:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport secrets\nimport hmac\nfrom datetime import datetime, timedelta\nfrom typing import Annotated\n\nimport bcrypt as _bcrypt_lib\nfrom fastapi import FastAPI, Depends, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.security import HTTPBearer, HTTPAuthorizationCredentials\nfrom jose import jwt, JWTError\nfrom pydantic import BaseModel, Field\nimport ollama\n\n# --- config ---------------------------------------------------------------\nSECRET_KEY = "change-me-in-production-use-env-var"\nALGORITHM = "HS256"\nTOKEN_EXPIRE_MINUTES = 60\nMODEL = "llama3.2"\n\n# --- password hashing -----------------------------------------------------\ndef hash_password(password: str) -> str:\n    return _bcrypt_lib.hashpw(password.encode(), _bcrypt_lib.gensalt()).decode()\n\ndef verify_password(plain: str, hashed: str) -> bool:\n    return _bcrypt_lib.checkpw(plain.encode(), hashed.encode())\n\n# --- JWT ------------------------------------------------------------------\ndef create_token(data: dict, expires_minutes: int = TOKEN_EXPIRE_MINUTES) -> str:\n    payload = {**data, "exp": datetime.utcnow() + timedelta(minutes=expires_minutes)}\n    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)\n\ndef decode_token(token: str) -> dict:\n    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])\n\n# --- in-memory user store (swap for SQLAlchemy in production) -------------\n_users: dict[str, dict] = {}        # email -> {id, email, hashed_password}\n_histories: dict[int, list] = {}    # user_id -> conversation messages\n_next_id: list[int] = [1]\n\n# --- Pydantic models ------------------------------------------------------\nclass RegisterRequest(BaseModel):\n    email: str = Field(min_length=3)\n    password: str = Field(min_length=6)\n\nclass LoginRequest(BaseModel):\n    email: str\n    password: str\n\nclass ChatRequest(BaseModel):\n    message: str = Field(min_length=1)\n\n# --- auth dependency ------------------------------------------------------\n_security = HTTPBearer()\n\ndef get_current_user(\n    creds: Annotated[HTTPAuthorizationCredentials, Depends(_security)]\n) -> dict:\n    try:\n        return decode_token(creds.credentials)\n    except JWTError:\n        raise HTTPException(status_code=401, detail="Invalid or expired token")\n\n# --- app ------------------------------------------------------------------\napp = FastAPI(title="Auth AI API", version="1.0")\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["http://localhost:8501"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n@app.post("/register", status_code=201)\ndef register(req: RegisterRequest):\n    if req.email in _users:\n        raise HTTPException(status_code=409, detail="Email already registered")\n    uid = _next_id[0]\n    _next_id[0] += 1\n    _users[req.email] = {\n        "id": uid,\n        "email": req.email,\n        "hashed_password": hash_password(req.password),\n    }\n    return {"id": uid, "email": req.email}\n\n@app.post("/login")\ndef login(req: LoginRequest):\n    user = _users.get(req.email)\n    if not user or not verify_password(req.password, user["hashed_password"]):\n        raise HTTPException(status_code=401, detail="Invalid credentials")\n    token = create_token({"user_id": user["id"], "email": req.email})\n    return {"access_token": token, "token_type": "bearer"}\n\n@app.get("/me")\ndef me(user: dict = Depends(get_current_user)):\n    return {"user_id": user["user_id"], "email": user["email"]}\n\n@app.post("/chat")\ndef chat(req: ChatRequest, user: dict = Depends(get_current_user)):\n    uid = user["user_id"]\n    hist = _histories.setdefault(uid, [])\n    hist.append({"role": "user", "content": req.message})\n    reply = ollama.chat(model=MODEL, messages=hist)["message"]["content"]\n    hist.append({"role": "assistant", "content": reply})\n    return {"reply": reply, "history_length": len(hist)}\n\nif __name__ == "__main__":\n    import uvicorn\n    uvicorn.run(app, host="0.0.0.0", port=8000)\n'

# ── write_auth_api ──────────────────────────────────────────────────────────
from pathlib import Path

def write_auth_api(path: str) -> str:
    Path(path).write_text(_AUTH_API_SRC, encoding="utf-8")
    return path

# ── generate the file ───────────────────────────────────────────────────────
out = write_auth_api("auth_api.py")
print(f"Generated: {out}  ({len(_AUTH_API_SRC)} chars)")
print(Path(out).read_text(encoding="utf-8")[:120] + "...")


In [ ]:
# ── smoke-test the auth flow with TestClient ────────────────────────────────
import bcrypt as _bcrypt_lib
from jose import jwt, JWTError
from datetime import datetime, timedelta
from typing import Annotated
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# --- replicate auth_api internals in-process (no ollama needed for checks) ---
SECRET_KEY = "change-me-in-production-use-env-var"
ALGORITHM  = "HS256"

def hash_password(p): return _bcrypt_lib.hashpw(p.encode(), _bcrypt_lib.gensalt()).decode()
def verify_password(plain, hashed): return _bcrypt_lib.checkpw(plain.encode(), hashed.encode())
def create_token(data, mins=60):
    pl = {**data, "exp": datetime.utcnow() + timedelta(minutes=mins)}
    return jwt.encode(pl, SECRET_KEY, algorithm=ALGORITHM)
def decode_token(tok): return jwt.decode(tok, SECRET_KEY, algorithms=[ALGORITHM])

_users: dict = {}
_histories: dict = {}
_nid = [1]

class RegReq(BaseModel):
    email: str = Field(min_length=3)
    password: str = Field(min_length=6)
class LoginReq(BaseModel):
    email: str; password: str

_sec = HTTPBearer()

def get_current_user(creds: Annotated[HTTPAuthorizationCredentials, Depends(_sec)]) -> dict:
    try:
        return decode_token(creds.credentials)
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")

app = FastAPI(title="Auth AI API (test)")

@app.post("/register", status_code=201)
def register(req: RegReq):
    if req.email in _users:
        raise HTTPException(409, "Email already registered")
    uid = _nid[0]; _nid[0] += 1
    _users[req.email] = {"id": uid, "email": req.email,
                          "hashed_password": hash_password(req.password)}
    return {"id": uid, "email": req.email}

@app.post("/login")
def login(req: LoginReq):
    u = _users.get(req.email)
    if not u or not verify_password(req.password, u["hashed_password"]):
        raise HTTPException(401, "Invalid credentials")
    return {"access_token": create_token({"user_id": u["id"], "email": req.email}),
            "token_type": "bearer"}

@app.get("/me")
def me(user: dict = Depends(get_current_user)):
    return {"user_id": user["user_id"], "email": user["email"]}

# ── run checks ──────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=False)
score = 0; total = 5

def chk(n, ok, msg):
    global score
    print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
    if ok: score += 1

# 1. register → 201
r = client.post("/register", json={"email": "alice@test.com", "password": "pass1234"})
chk(1, r.status_code == 201, f"POST /register → 201 (got {r.status_code})")

# 2. duplicate → 409
r2 = client.post("/register", json={"email": "alice@test.com", "password": "pass1234"})
chk(2, r2.status_code == 409, f"duplicate email → 409 (got {r2.status_code})")

# 3. login → access_token
r3 = client.post("/login", json={"email": "alice@test.com", "password": "pass1234"})
chk(3, r3.status_code == 200 and "access_token" in r3.json(),
    f"POST /login → access_token (got {r3.status_code})")

tok = r3.json().get("access_token", "")

# 4. /me with token → user_id
r4 = client.get("/me", headers={"Authorization": f"Bearer {tok}"})
chk(4, r4.status_code == 200 and r4.json().get("email") == "alice@test.com",
    f"GET /me → email (got {r4.status_code})")

# 5. /me without token → 401 (HTTPBearer auto-rejects missing header)
r5 = client.get("/me")
chk(5, r5.status_code == 401, f"GET /me no token → 401 (got {r5.status_code})")

print(f"\nScore: {score} / {total}")
if score == total:
    print("\nDay 55 — Authentication complete! 🎉")
print(f"\nDeliverable: auth_api.py generated ({len(_AUTH_API_SRC)} chars)")
print("Run:  uvicorn auth_api:app --reload")
print("Docs: http://localhost:8000/docs")
